In [ ]:
# Install dependencies if needed
# !pip install -q -r ../requirements.txt


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
              

In [ ]:
from pathlib import Path

def find_submission_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "model" / "best_adapter").exists() and (path / "data" / "test.csv").exists():
            return path
    # When running this notebook from w16/poc/app, parents[1] is w16.
    notebook_guess = Path.cwd().resolve().parents[1]
    if (notebook_guess / "model" / "best_adapter").exists():
        return notebook_guess
    raise FileNotFoundError("Cannot find submission root containing model/best_adapter and data/test.csv")

SUBMISSION_ROOT = find_submission_root()
adapter_path = SUBMISSION_ROOT / "model" / "best_adapter"
test_csv_path = SUBMISSION_ROOT / "data" / "test.csv"
base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"Submission root: {SUBMISSION_ROOT}")
print(f"Adapter path: {adapter_path}")
print(f"Test CSV: {test_csv_path}")


Mounted at /content/drive


In [ ]:
import torch
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(adapter_path, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, adapter_path, local_files_only=True)
model.eval()
print("Model and adapter loaded.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:03<?, ?it/s]

In [ ]:
import pandas as pd

if test_csv_path.exists():
    df = pd.read_csv(test_csv_path)
    print(f"Loaded test data: {len(df)} rows")
else:
    df = None
    print(f"Missing test data: {test_csv_path}")


已讀取雲端資料夾內的測試資料


In [ ]:
#模型資安推理與動態控制邏輯
def analyze_and_control(email_text, ui_threshold):
    if not email_text or not email_text.strip():
        return 0.0, "無", "⚠️ 錯誤", "請先輸入測試文本（主旨或內文）"

    messages = [
        {
            "role": "system",
            "content": "You are a cybersecurity email classifier. Decide whether the email is phishing or legitimate. Do not copy the email. Do not explain. Output only one label: phishing or legitimate."
        },
        {
            "role": "user",
            "content": f"Body: {email_text}"
        }
    ]

    # 模型生成標籤
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(**model_inputs, max_new_tokens=10)

    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    ai_response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip().lower()

    # 動態控制決策
    if "phishing" in ai_response:
        risk_score = 1.0
        risk_level = "HIGH (高風險)"
        control_action = "🚨 攔截並發出警報 (BLOCK) —— 專屬模型判定為 Phishing 威脅"
        details = f"【模型偵測結果】: {ai_response}\n【資安決策】: 模型認定此郵件包含高度商業郵件詐騙 (BEC) 特徵，系統已強制阻斷。"

    elif "legitimate" in ai_response:
        risk_score = 0.0
        risk_level = "LOW (低風險)"
        control_action = "🟢 安全放行 (PASS) —— 專屬模型判定為 Legitimate 安全"
        details = f"【模型偵測結果】: {ai_response}\n【資安決策】: 模型評估此郵件無可疑社交工程特徵，准予放行至員工收件匣。"

    else:
        # 異常狀態下的管理員門檻控制政策
        risk_score = 0.5
        risk_level = "UNKNOWN (無法辨識)"
        if ui_threshold <= 0.5:
            control_action = f"🚨 攔截控制 (BLOCK) —— 模型輸出異常({ai_response})，依防禦政策強制阻斷！"
            details = f"【異常處理】: 模型無法給出明確標籤。因為網頁端設定的門檻為 {ui_threshold} (屬嚴格政策)，資安系統採取『寧可錯殺』方針。"
        else:
            control_action = f"🟢 異常放行 (PASS) —— 模型輸出異常({ai_response})，依防禦政策容忍放行。"
            details = f"【異常處理】: 模型無法給出明確標籤。因為網頁端設定的門檻為 {ui_threshold} (屬寬鬆政策)，系統選擇放行。"

    return risk_score, risk_level, control_action, details




In [ ]:
def load_sample_by_index(index):
    if df is not None:
        if 0 <= index < len(df):
            row = df.iloc[index]

            # 抓取主旨
            subject = "無主旨"
            for col in ['subject', 'Subject', 'title']:
                if col in df.columns:
                    subject = str(row[col])
                    break


            body = "無內文"
            for col in ['email_text', 'body', 'text', 'content']:
                if col in df.columns:
                    body = str(row[col])
                    break

            # 如果 CSV 裡真的沒有專門的內文欄位，自動把主旨代入分析
            if body == "無內文" and subject != "無主旨":
                body = f"[系統智慧分析模式：已由主旨自動代入]\n{subject}"

            return subject, body
        else:
            return "索引超出範圍", f"目前資料筆數僅有 {len(df)} 筆 (0 到 {len(df)-1})"
    return "找不到主旨", "找不到對應樣本，請確認 test.csv 是否存在於 /content/ 目錄。"


In [ ]:
import gradio as gr

with gr.Blocks(title="BEC 防禦控制系統") as demo:
    gr.Markdown("# 基於專屬微調 Qwen2.5 之 BEC 詐騙郵件動態控制防禦系統")
    gr.Markdown("系統已載入提交包中的 **LoRA adapter** 與測試資料，可進行本機或 Colab PoC 展示。")


    with gr.Row():
        # 區塊一 (左上)
        with gr.Column(variant="panel"):
            gr.Markdown("### 📥 區塊一：測試數據輸入")
            with gr.Row():
                sample_index = gr.Number(value=0, label="測試集索引 (Index)", precision=0)
                load_btn = gr.Button("📂 載入樣本", variant="secondary")
            input_subject = gr.Textbox(label="郵件主旨 (Subject)", placeholder="自動載入或自行輸入...")
            input_body = gr.Textbox(label="郵件內文 (Body) —— 已啟用智慧主旨分析機制", lines=5, placeholder="請輸入電子郵件內容...")

        # 區塊二 (右上)
        with gr.Column(variant="panel"):
            gr.Markdown("### 📊 區塊二：模型風險數據")
            output_score = gr.Number(label="Risk Score (客觀風險分數參考)", interactive=False)
            output_level = gr.Textbox(label="Risk Level (風險等級)", interactive=False)

    with gr.Row():
        # 區塊三 (左下)
        with gr.Column(variant="panel"):
            gr.Markdown("### ⚙️ 區塊三：防禦原則動態設定")
            threshold_slider = gr.Slider(minimum=0.0, maximum=1.0, value=0.5, step=0.05, label="防禦攔截門檻 (ui_threshold)")
            submit_btn = gr.Button("⚡ 啟動模型偵測與控制輸出", variant="primary")

        # 區塊四 (右下)
        with gr.Column(variant="panel"):
            gr.Markdown("### 🛡️ 區塊四：最終防禦控制動作")
            output_action = gr.Textbox(label="最終控制動作 (Control Action)", interactive=False)
            output_details = gr.TextArea(label="📝 詳細決策與異常處理說明", lines=4, interactive=False)

    load_btn.click(
        fn=load_sample_by_index,
        inputs=[sample_index],
        outputs=[input_subject, input_body]
    )

    submit_btn.click(
        fn=analyze_and_control,
        inputs=[input_body, threshold_slider],
        outputs=[output_score, output_level, output_action, output_details]
    )


print("正在啟動 Gradio 資安網頁服務，請點擊下方產生的 public URL 連結開啟網頁...")
demo.launch(share=True, debug=True)

正在啟動 Gradio 資安網頁服務，請點擊下方產生的 public URL 連結開啟網頁...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e9f5d9450207f65a1a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
